In [1]:
!pip install motmetrics filterpy lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.7 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 40.0 MB/s eta 0:00:0000:01
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=88aeb28f51753b539f7af3d28e9b311fc67177d41bbc6dd5eeb9f25b474fe48d
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [2]:
import os
import json
import cv2
import matplotlib.pyplot as plt
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

# Hàm bổ trợ

In [3]:
import pandas as pd
import numpy as np
import os
import glob


TARGET_CLASSES = ['Car', 'Pedestrian', 'Cyclist']
def parse_kitti_tracking_file(label_path):
    """
    Đọc file label chuẩn KITTI Tracking (0000.txt)
    Format: frame id type truncated occluded alpha x1 y1 x2 y2 h w l x y z rot_y
    """
    if not os.path.exists(label_path):
        print(f"File not found: {label_path}")
        return {}, pd.DataFrame()
    # Đọc file space-separated
    columns = [
        'frame', 'track_id', 'type', 'truncated', 'occluded', 'alpha', 
        'bbox_l', 'bbox_t', 'bbox_r', 'bbox_b', 
        'dim_h', 'dim_w', 'dim_l', 'loc_x', 'loc_y', 'loc_z', 'rot_y'
    ]
    df = pd.read_csv(label_path, sep=' ', names=columns)
    df = df[df['track_id'] != -1]
    df = df[df['type'].isin(TARGET_CLASSES)]
    
    # 1. Chuẩn bị Detections (Input cho Tracker)
    # Gom nhóm theo frame để dễ truy xuất trong vòng lặp
    detections = {}
    for frame_idx, group in df.groupby('frame'):
        dets_list = []
        for _, row in group.iterrows():
            # Ở đây ta dùng GT làm Detection để test logic code trước (Oracle Test)
            # Bạn có thể thêm nhiễu (noise) vào bbox để test độ bền của Tracker
            det = {
                'boxs': [row['bbox_l'], row['bbox_t'], row['bbox_r'], row['bbox_b']],
                'score': 0.99,'label': row['type']
            }
            dets_list.append(det)
        detections[frame_idx] = dets_list
    # 2. Chuẩn bị Ground Truth (Output cho Evaluator)
    # Trả về DataFrame gốc để motmetrics xử lý
    return detections, df

In [4]:
def convert_bbox(bbox):
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = bbox[0] + w/2.
    y = bbox[1] + h/2.
    s = w * h
    r = w / float(h) if h > 0 else 1.0
    return np.array([[x], [y], [s], [r]]).reshape((4, 1))

def get_box_coords(state):
    """
    Chuyển từ [x, y, s, r] về [x1, y1, x2, y2] để tính IOU
    """
    x, y, s, r = state
    w = np.sqrt(s * r)
    h = s / w
    x1 = x - w / 2.
    y1 = y - h / 2.
    x2 = x + w / 2.
    y2 = y + h / 2.
    return [x1, y1, x2, y2]

def calculate_iou(box1, box2):
    """
    box1, box2: [x1, y1, x2, y2]
    """
    xx1 = max(box1[0], box2[0])
    yy1 = max(box1[1], box2[1])
    xx2 = min(box1[2], box2[2])
    yy2 = min(box1[3], box2[3])

    w = max(0, xx2 - xx1)
    h = max(0, yy2 - yy1)
    inter = w * h
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def filter_small_objects(detections, img_w, img_h, min_ratio):
    """
    Lọc các detection có diện tích nhỏ hơn min_ratio * diện tích ảnh.
    """
    img_area = img_w * img_h
    min_area = img_area * min_ratio
    
    valid_detections = []
    
    for det in detections:
        box = det['boxs'] # [x1, y1, x2, y2]
        w = box[2] - box[0]
        h = box[3] - box[1]
        area = w * h
        
        # Chỉ giữ lại nếu diện tích đủ lớn
        if area >= min_area:
            valid_detections.append(det)
            
    return valid_detections

def compute_color_hist(image, bbox):
    """
    Tính đặc trưng màu sắc (HSV) của một bounding box.
    """
    h_img, w_img = image.shape[:2]
    x1, y1, x2, y2 = map(int, bbox)
    
    # Clip coordinates để không lỗi biên
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w_img, x2); y2 = min(h_img, y2)
    
    crop = image[y1:y2, x1:x2]
    if crop.size == 0: return None

    # Chuyển sang HSV
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    # Chỉ lấy Hue và Saturation
    hist = cv2.calcHist([hsv], [0, 1], None, [18, 20], [0, 180, 0, 256])
    
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist

def get_motion_info(trk):
    """Tính toán vận tốc và xác định hướng di chuyển"""
    vx = trk.kf.x[4][0]
    vy = trk.kf.x[5][0]
    speed = np.sqrt(vx**2 + vy**2)
    angle = np.degrees(np.arctan2(vy, vx))
    direction_str = "---"
        
    return speed, vx, vy, angle

In [5]:
def create_json_entry(folder_id, frame_id, det_info, tracker=None):
    entry = det_info.copy()
    entry["folder_id"] = str(folder_id)
    entry["frame_id"] = int(frame_id)

    if tracker is not None:
        vx, vy = getattr(tracker, "velocity_smooth", (0.0, 0.0))
        speed, vx, vy, angle= get_motion_info(tracker)

        entry["tracking_info"] = {
            "track_id": int(tracker.id),
            "velocity_vector": [round(float(vx), 2), round(float(vy), 2)],
            "speed_px": round(float(speed), 2),
            "angle_deg": round(float(angle), 1)
            # "direction": direct
        }
    else:
        entry["tracking_info"] = {
            "track_id": -1,
            "velocity_vector": [0.0, 0.0],
            "speed_px": 0.0,
            "angle_deg": 0.0
            # "direction": "New/Unknown"
        }

    return entry

def print_tabular_report(frame_idx, json_results):
    print(f"\n>>> BÁO CÁO FRAME {frame_idx}")
    print(f"{'ID':<4} | {'Label':<10} | {'Vận tốc (vx, vy)':<18} | {'Speed':<6} | {'Góc':<6}")
    print("-" * 80)
    
    for item in json_results:
        info = item["tracking_info"]
        vx, vy = info["velocity_vector"]
        print(f"{info['track_id']:<4} | {item['label']:<10} | ({vx:>6.1f}, {vy:>6.1f})    | {info['speed_px']:<6.1f} | {info['angle_deg']:<6.0f} ")
    print("=" * 80)

# Kalman + Association

In [6]:
import numpy as np
from filterpy.kalman import KalmanFilter

class Tracker:
    def __init__(self, track_id, initial_box, label, initial_image): # <--- THÊM initial_image
        self.id = track_id
        self.label = label
        self.time_since_update = 0
        self.hits = 0 
        self.velocity_smooth = np.array([0.0, 0.0]) 
        self.hist = None

        box = convert_bbox(initial_box) 
        # Dùng .item() chuẩn hơn float() cho numpy array
        self.last_pos = np.array([box[0].item(), box[1].item()]) 

        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        dt = 1 
        
        self.kf.F = np.array([
            [1, 0, 0, 0, dt, 0, 0], 
            [0, 1, 0, 0, 0, dt, 0], 
            [0, 0, 1, 0, 0, 0, dt],
            [0, 0, 0, 1, 0, 0, 0],  
            [0, 0, 0, 0, 1, 0, 0],  
            [0, 0, 0, 0, 0, 1, 0], 
            [0, 0, 0, 0, 0, 0, 1]
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0], 
            [0, 1, 0, 0, 0, 0, 0], 
            [0, 0, 1, 0, 0, 0, 0], 
            [0, 0, 0, 1, 0, 0, 0]
        ])
        
        self.kf.R[2:, 2:] *= 10.  
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P *= 10.
        self.kf.Q[4:, 4:] *= 0.01
        self.kf.x[:4] = convert_bbox(initial_box)

        # Tính toán histogram ban đầu
        # Lưu ý: Hàm compute_color_hist phải được định nghĩa ở ngoài class này
        self.hist = compute_color_hist(initial_image, initial_box)


    def apply_camera_motion(self, M):
        """
        Dịch chuyển vị trí dự đoán theo chuyển động của Camera
        """
        cx = self.kf.x[0].item()
        cy = self.kf.x[1].item()
        pos = np.array([cx, cy, 1.0]) 
        new_pos = M @ pos 
        self.kf.x[0] = new_pos[0]
        self.kf.x[1] = new_pos[1]
        
        last_pos_vec = np.array([self.last_pos[0], self.last_pos[1], 1.0])
        new_last_pos = M @ last_pos_vec
        
        self.last_pos[0] = new_last_pos[0]
        self.last_pos[1] = new_last_pos[1]

    def predict(self):
        if (self.kf.x[6].item() + self.kf.x[2].item()) <= 0: 
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.time_since_update += 1

    def update(self, bbox, raw_det=None, image=None): # <--- THÊM image=None
        self.time_since_update = 0
        self.hits += 1      
        self.kf.update(convert_bbox(bbox)) 

        # --- HIỂN THỊ VẬN TỐC & HƯỚNG ---
        raw_vx = self.kf.x[4].item()
        raw_vy = self.kf.x[5].item()
        
        # EMA smoothing
        alpha = 0.15
        if self.hits <= 2:
            self.velocity_smooth = np.array([raw_vx, raw_vy])
        else:
            self.velocity_smooth = alpha * np.array([raw_vx, raw_vy]) + \
                                   (1 - alpha) * self.velocity_smooth
        
        self.last_pos = np.array([self.kf.x[0].item(), self.kf.x[1].item()])
        
        if raw_det is not None:
            self.last_raw_det = raw_det

        # --- CẬP NHẬT HISTOGRAM (MỚI) ---
        if image is not None:
            # Sửa tên hàm cho đồng nhất: compute_color_hist
            new_hist = compute_color_hist(image, bbox) 
            
            if self.hist is None:
                self.hist = new_hist
            elif new_hist is not None:
                # Cập nhật mềm: giữ 70% cũ, nạp 30% mới
                self.hist = 0.7 * self.hist + 0.3 * new_hist
                # QUAN TRỌNG: Chuẩn hóa lại về dải [0, 1]
                cv2.normalize(self.hist, self.hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)


In [7]:
def mahalanobis_distance(trk, det_cx, det_cy):
    z = np.array([det_cx, det_cy])
    x = trk.kf.x[:2].flatten()
    P = trk.kf.P[:2, :2]
    S = P + trk.kf.R[:2, :2]  # innovation covariance
    diff = z - x
    return np.sqrt(diff.T @ np.linalg.inv(S) @ diff)

In [8]:
def associate_object(detections, trackers, current_image):
    if len(trackers) == 0:
        return np.empty((0, 2), dtype=int), list(range(len(detections))), []

    # =========================================================
    # VÒNG 1: HIGH CONFIDENCE (Dựa trên IoU - Strict)
    # =========================================================
    iou_matrix = np.zeros((len(trackers), len(detections)))
    for t, trk in enumerate(trackers):
        trk_rect = get_box_coords(trk.kf.x.flatten()[:4])
        for d, det in enumerate(detections):
            if trk.label != det['label']:
                iou_matrix[t, d] = 1.0 
                continue
            iou = calculate_iou(trk_rect, det['boxs'])
            iou_matrix[t, d] = 1.0 - iou

    row_ind, col_ind = linear_sum_assignment(iou_matrix)

    matches_1 = []
    matched_trk_idx = set()
    matched_det_idx = set()
    iou_threshold = 0.8  # IoU phải > 0.2

    for r, c in zip(row_ind, col_ind):
        if iou_matrix[r, c] < iou_threshold:
            matches_1.append([r, c])
            matched_trk_idx.add(r)
            matched_det_idx.add(c)

    unmatched_trks = [t for t in range(len(trackers)) if t not in matched_trk_idx]
    unmatched_dets = [d for d in range(len(detections)) if d not in matched_det_idx]


    # =========================================================
    # VÒNG 2: RECOVERY (MÀU SẮC + KHOẢNG CÁCH XA)
    # =========================================================
    matches_2 = []

    if len(unmatched_trks) > 0 and len(unmatched_dets) > 0:
        cost_matrix_2 = np.zeros((len(unmatched_trks), len(unmatched_dets)))

        for i, t_idx in enumerate(unmatched_trks):
            trk = trackers[t_idx]
            t_box = get_box_coords(trk.kf.x.flatten()[:4])
            t_cx, t_cy = (t_box[0]+t_box[2])/2, (t_box[1]+t_box[3])/2
            
            for j, d_idx in enumerate(unmatched_dets):
                det = detections[d_idx]
                if trk.label != det['label']:
                    cost_matrix_2[i, j] = 1e9
                    continue

                # 1. Cost Màu
                det_hist = compute_color_hist(current_image, det['boxs'])
                d_color = 1.0
                if trk.hist is not None and det_hist is not None:
                    d_color = cv2.compareHist(trk.hist, det_hist, cv2.HISTCMP_BHATTACHARYYA)

                # 2. Cost Khoảng Cách
                d_box = det['boxs']
                d_cx, d_cy = (d_box[0]+d_box[2])/2, (d_box[1]+d_box[3])/2
                d_h = d_box[3] - d_box[1]
                dist_px = np.sqrt((t_cx - d_cx)**2 + (t_cy - d_cy)**2)
                norm_dist = dist_px / d_h if d_h > 0 else 100.0

                # 3. Cost Mahalanobis
                d_maha = mahalanobis_distance(trk, d_cx, d_cy)

                # GATING
                if d_color > 0.4 or norm_dist > 3.0 or d_maha > 3.0:
                    cost_matrix_2[i, j] = 1e5
                else:
                    cost_matrix_2[i, j] = 0.7 * d_color + 0.2 * (norm_dist / 3.0) + 0.1 * (d_maha / 3.0)

        r_ind_2, c_ind_2 = linear_sum_assignment(cost_matrix_2)
        
        for r, c in zip(r_ind_2, c_ind_2):
            if cost_matrix_2[r, c] < 0.5:
                matches_2.append([unmatched_trks[r], unmatched_dets[c]])

    # =========================================================
    # TỔNG HỢP
    # =========================================================
    final_matches = list(matches_1) + matches_2
    final_matches = np.array(final_matches) if len(final_matches) > 0 else np.empty((0, 2), dtype=int)
    
    if len(final_matches) > 0:
        matched_trk_set = set(final_matches[:, 0])
        matched_det_set = set(final_matches[:, 1])
    else:
        matched_trk_set = set()
        matched_det_set = set()

    final_unmatched_dets = [d for d in range(len(detections)) if d not in matched_det_set]
    final_unmatched_trks = [t for t in range(len(trackers)) if t not in matched_trk_set]

    return final_matches, final_unmatched_dets, final_unmatched_trks



In [9]:
class CameraMotionEstimator:
    def __init__(self):
        self.detector = cv2.ORB_create(nfeatures=500)
        self.matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        self.prev_kp = None
        self.prev_des = None
        
    def estimate(self, curr_frame):
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        curr_kp, curr_des = self.detector.detectAndCompute(curr_gray, None)

        if self.prev_kp is None or curr_des is None or len(curr_kp) < 20:
            self.prev_kp = curr_kp
            self.prev_des = curr_des
            return np.eye(3)

        matches = self.matcher.match(self.prev_des, curr_des)
        src_pts = []
        dst_pts = []
        for m in matches:
            src_pts.append(self.prev_kp[m.queryIdx].pt)
            dst_pts.append(curr_kp[m.trainIdx].pt)

        src_pts = np.float32(src_pts).reshape(-1, 1, 2)
        dst_pts = np.float32(dst_pts).reshape(-1, 1, 2)

        if len(matches) > 10:
            M, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC)
            if M is None: 
                M_3x3 = np.eye(3)
            else:
                M_3x3 = np.eye(3)
                M_3x3[:2] = M
        else:
            M_3x3 = np.eye(3)

        self.prev_kp = curr_kp
        self.prev_des = curr_des
        return M_3x3

In [10]:
import motmetrics as mm
import cv2
from tqdm.notebook import tqdm # Thanh tiến trình cho đẹp
import matplotlib.pyplot as plt

# --- FIX LỖI NUMPY 2.0 ---
# Định nghĩa lại hàm asfarray đã bị xóa bằng cách trỏ nó về asarray(..., dtype=float)
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda x: np.asarray(x, dtype=float)
# -------------------------

# Test seq0003

In [11]:
# # --- CẤU HÌNH ĐƯỜNG DẪN KAGGLE ---
# KAGGLE_ROOT = "/kaggle/input/datasets/leducnhuan/kitti-tracking/kitti_tracking/training/image_02"
# LABEL_ROOT = "/kaggle/input/datasets/leducnhuan/kitti-tracking/kitti_tracking/training/label_02"

# # Chọn sequence để test
# SEQ_ID = "0003"

# img_dir = os.path.join(KAGGLE_ROOT, SEQ_ID)
# label_file = os.path.join(LABEL_ROOT, f"{SEQ_ID}.txt")

# # 1. Load dâta
# print(f"Loading data for Sequence {SEQ_ID}...")
# seq_detections, gt_df = parse_kitti_tracking_file(label_file)
# image_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))

# if len(image_paths) == 0:
#     print("Lỗi: Không tìm thấy ảnh. Hãy kiểm tra lại đường dẫn KAGGLE_ROOT")
# else:
#     print(f"Đã tải {len(image_paths)} frames.")

# # 2. Khởi tạo metrics
# acc = mm.MOTAccumulator(auto_id=True)

# # 3. KHỞI TẠO TRACKER
# # TAU_HIGH = 0.6
# # TAU_LOW = 0.1
# MAX_LOST_FRAMES = 30 # số frames giữ tracker bị mất trước khi xóa hẳn

# tracker_list = []
# track_id_count = 1
# motion_estimator = CameraMotionEstimator()

# print("\n>>> Start Tracking...")

# for k, img_path in enumerate(tqdm(image_paths)):
#     img = cv2.imread(img_path)
#     if img is None:
#         continue

#     # =========================
#     # 1. GET DETECTIONS
#     # =========================
#     frame_dets = seq_detections.get(k, [])

#     # =========================
#     # 2. PREDICT + CAMERA MOTION
#     # =========================
#     M = motion_estimator.estimate(img)
#     for trk in tracker_list:
#         trk.predict()
#         trk.apply_camera_motion(M)

#     # =========================
#     # 3. ASSOCIATION
#     # =========================
#     matches, unmatched_dets, unmatched_trks = associate_object(
#         detections=frame_dets,
#         trackers=tracker_list,
#         current_image=img
#     )

#     # =========================
#     # 4. UPDATE MATCHED
#     # =========================
#     matched_trk_ids = set()
#     for trk_idx, det_idx in matches:
#         trk = tracker_list[trk_idx]
#         det = frame_dets[det_idx]
#         trk.update(det['boxs'], raw_det=det, image=img)
#         matched_trk_ids.add(trk.id)

#     # =========================
#     # 5. CREATE NEW TRACKS
#     # =========================
#     for det_idx in unmatched_dets:
#         det = frame_dets[det_idx]
#         new_trk = Tracker(track_id_count, det['boxs'], det['label'], initial_image=img)
#         tracker_list.append(new_trk)
#         track_id_count += 1

#     # =========================
#     # 6. REMOVE DEAD TRACKS
#     # =========================
#     tracker_list = [t for t in tracker_list if t.time_since_update <= MAX_LOST_FRAMES]

#     # =========================
#     # 7. COLLECT HYPOTHESIS (ONLY MATCHED)
#     # =========================
#     frame_hypos = []
#     frame_hypo_ids = []

#     for trk in tracker_list:
#         if trk.time_since_update == 0:  # chỉ lấy tracker vừa match
#             box = get_box_coords(trk.kf.x.flatten()[:4])

#             x1, y1, x2, y2 = box
            
#             w = x2 - x1
#             h = y2 - y1
#             frame_hypos.append([x1, y1, w, h])
#             frame_hypo_ids.append(trk.id)

#     # =========================
#     # 8. COLLECT GT
#     # =========================
#     frame_gt = []
#     frame_gt_ids = []

#     if not gt_df.empty:
#         gt_now = gt_df[gt_df['frame'] == k]
#         for _, row in gt_now.iterrows():
#             if int(row['track_id']) == -1:
#                 continue
#             w = row['bbox_r'] - row['bbox_l']
#             h = row['bbox_b'] - row['bbox_t']
#             frame_gt.append([row['bbox_l'], row['bbox_t'], w, h])
#             frame_gt_ids.append(int(row['track_id']))

#     # =========================
#     # 9. UPDATE METRICS
#     # =========================
#     if not hasattr(np, 'asfarray'):
#         np.asfarray = lambda x: np.asarray(x, dtype=float)

#     dist_matrix = mm.distances.iou_matrix(frame_gt, frame_hypos, max_iou=0.5)
#     acc.update(frame_gt_ids, frame_hypo_ids, dist_matrix)

# print("Tracking Completed.")

In [12]:
# print(f"\n{'='*20} EVALUATION REPORT FOR SEQ {SEQ_ID} {'='*20}")

# mh = mm.metrics.create()
# # Các chỉ số quan trọng: MOTA, IDF1 (độ ổn định ID), MOTP (độ khớp box)
# summary = mh.compute(acc, metrics=['num_frames', 'mota', 'motp', 'idf1', 'idp', 'idr', 'num_switches'], name='OriginalAssociation')

# strsummary = mm.io.render_summary(
#     summary,
#     formatters=mh.formatters,
#     namemap=mm.io.motchallenge_metric_names
# )
# print(strsummary)

# Scale lên toàn data

In [21]:
import time

# --- PATH ---
KAGGLE_ROOT = "/kaggle/input/datasets/leducnhuan/kitti-tracking/kitti_tracking/training/image_02"
LABEL_ROOT = "/kaggle/input/datasets/leducnhuan/kitti-tracking/kitti_tracking/training/label_02"

# Lấy danh sách sequence (0000,0001,...)
SEQ_LIST = sorted(os.listdir(KAGGLE_ROOT))
print("Sequences:", SEQ_LIST)

all_accs = {c: [] for c in TARGET_CLASSES}
all_names = {c: [] for c in TARGET_CLASSES}

motion_estimator = CameraMotionEstimator()

tracking_time = 0
tracking_calls = 0

# =========================
# LOOP ALL SEQUENCES
# =========================

for SEQ_ID in SEQ_LIST:
    print(f"\n========== Processing Sequence {SEQ_ID} ==========")
    
    img_dir = os.path.join(KAGGLE_ROOT, SEQ_ID)
    label_file = os.path.join(LABEL_ROOT, f"{SEQ_ID}.txt")

    if not os.path.exists(label_file):
        print(f" Skip {SEQ_ID}, no label file")
        continue

    # Load detections & GT
    seq_detections, gt_df = parse_kitti_tracking_file(label_file)
    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
    print(f"Frames: {len(image_paths)}")

    # =========================
    # RESET TRACKERS PER SEQ
    # =========================
    
    # Tạo danh sách tracker và đếm ID riêng biệt cho từng class
    trackers = {c: [] for c in TARGET_CLASSES}
    track_id_counts = {c: 1 for c in TARGET_CLASSES}
    accs = {c: mm.MOTAccumulator(auto_id=True) for c in TARGET_CLASSES}

    # # =========================
    # # FRAME LOOP
    # # =========================
    # for k, img_path in enumerate(tqdm(image_paths, leave=False)):
    #     img = cv2.imread(img_path)
    #     if img is None:
    #         continue
    #     img_h, img_w = img.shape[0], img.shape[1]
    #     # Ước lượng chuyển động Camera 1 lần cho toàn bộ frame
    #     start_cmc = time.time()
    #     M = motion_estimator.estimate(img)
    #     tracking_time += time.time() - start_cmc
    #     # Lấy toàn bộ detections của frame hiện tại
    #     frame_dets_all_classes = seq_detections.get(k, {c: [] for c in TARGET_CLASSES})

    #     for class_name in TARGET_CLASSES:
    #         # --- Chuẩn bị Detections ---
    #         valid_dets = []
    #         for det in frame_dets_all_classes[class_name]:
    #             box = det['boxs']
    #             # Clip box vào trong viền ảnh
    #             x1 = max(0, min(box[0], img_w - 1))
    #             y1 = max(0, min(box[1], img_h - 1))
    #             x2 = max(0, min(box[2], img_w - 1))
    #             y2 = max(0, min(box[3], img_h - 1))
    #             w = x2 - x1
    #             h = y2 - y1
                
    #             # Bỏ qua box bị âm hoặc diện tích bằng 0
    #             if w <= 0 or h <= 0:
    #                 continue
                
    #             # Đóng gói lại đúng cấu trúc cho thuật toán tự xây
    #             valid_dets.append({
    #                 'boxs': [x1, y1, x2, y2],
    #                 'score': det.get('score', 1.0),
    #                 'label': class_name
    #             })
    #         # --- 2. BẮT ĐẦU ĐO THỜI GIAN TRACKING LOGIC ---
    #         start_track = time.time()
    #         tracker_list = trackers[class_name]
    #         for trk in tracker_list:
    #             trk.predict()
    #             trk.apply_camera_motion(M)
    #             # --- Association ---
    #         matches, unmatched_dets, unmatched_trks = associate_object(
    #             detections=valid_dets,
    #             trackers=tracker_list,
    #             current_image=img
    #         )
    #         # --- Update ---
    #         for trk_idx, det_idx in matches:
    #             trk = tracker_list[trk_idx]
    #             det = valid_dets[det_idx]
    #             trk.update(det['boxs'], raw_det=det, image=img)
    #         # --- Create new ---
    #         for det_idx in unmatched_dets:
    #             det = valid_dets[det_idx]
    #             new_trk = Tracker(track_id_counts, det['boxs'], det['label'], initial_image=img)
    #             tracker_list.append(new_trk)
    #             track_id_counts += 1
    #         # --- Remove dead ---
    #         trackers[class_name] = [t for t in tracker_list if t.time_since_update <= MAX_LOST_FRAMES]
    #         tracker_list = trackers[class_name]
            
    #         # CHỐT THỜI GIAN
    #         tracking_time += time.time() - start_track
    #         tracking_calls += 1
            
    #         # =========================
    #         # COLLECT HYPOTHESIS
    #         # =========================
    #         frame_hypos = []
    #         frame_hypo_ids = []
            
    #         for trk in tracker_list:
    #             if trk.time_since_update == 0:
    #                 box = get_box_coords(trk.kf.x.flatten()[:4])
    #                 x1, y1, x2, y2 = box
    #                 w = x2 - x1
    #                 h = y2 - y1
    #                 frame_hypos.append([x1, y1, w, h])
    #                 frame_hypo_ids.append(trk.id)

    #         # =========================
    #         # COLLECT GT
    #         # =========================
    #         frame_gt = []
    #         frame_gt_ids = []
    #         if not gt_df.empty:
    #             gt_now = gt_df[(gt_df['frame'] == k) & (gt_df['type'] == class_name)]
    #             for _, row in gt_now.iterrows():
    #                 if int(row['track_id']) == -1:
    #                     continue
    #                 w = row['bbox_r'] - row['bbox_l']
    #                 h = row['bbox_b'] - row['bbox_t']
    #                 frame_gt.append([row['bbox_l'], row['bbox_t'], w, h])
    #                 frame_gt_ids.append(int(row['track_id']))

    #         # =========================
    #         # UPDATE METRICS
    #         # =========================
    #         if not hasattr(np, 'asfarray'):
    #             np.asfarray = lambda x: np.asarray(x, dtype=float)

    #         dist_matrix = mm.distances.iou_matrix(frame_gt, frame_hypos, max_iou=0.5)
    #         # Sửa: Gọi accs theo class_name
    #         accs[class_name].update(frame_gt_ids, frame_hypo_ids, dist_matrix)
    
    
    # =========================
    # FRAME LOOP
    # =========================
    for k, img_path in enumerate(tqdm(image_paths, leave=False)):
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        img_h, img_w = img.shape[0], img.shape[1]
        
        # Ước lượng chuyển động Camera 1 lần cho toàn bộ frame
        start_cmc = time.time()
        M = motion_estimator.estimate(img)
        tracking_time += time.time() - start_cmc
        
        # SỬA LỖI 1: Lấy toàn bộ detections dưới dạng LIST
        frame_dets_all = seq_detections.get(k, [])

        for class_name in TARGET_CLASSES:
            # --- Chuẩn bị Detections ---
            valid_dets = []
            
            # Lọc các detections thuộc về class hiện tại
            class_dets = [d for d in frame_dets_all if d.get('label') == class_name]
            
            for det in class_dets:
                box = det['boxs']
                # Clip box vào trong viền ảnh
                x1 = max(0, min(box[0], img_w - 1))
                y1 = max(0, min(box[1], img_h - 1))
                x2 = max(0, min(box[2], img_w - 1))
                y2 = max(0, min(box[3], img_h - 1))
                w = x2 - x1
                h = y2 - y1
                
                # Bỏ qua box bị âm hoặc diện tích bằng 0
                if w <= 0 or h <= 0:
                    continue
                
                # Đóng gói lại đúng cấu trúc cho thuật toán tự xây
                valid_dets.append({
                    'boxs': [x1, y1, x2, y2],
                    'score': det.get('score', 1.0),
                    'label': class_name
                })
                
            # --- 2. BẮT ĐẦU ĐO THỜI GIAN TRACKING LOGIC ---
            start_track = time.time()
            tracker_list = trackers[class_name]
            
            for trk in tracker_list:
                trk.predict()
                trk.apply_camera_motion(M)
                
            # --- Association ---
            matches, unmatched_dets, unmatched_trks = associate_object(
                detections=valid_dets,
                trackers=tracker_list,
                current_image=img
            )
            
            # --- Update ---
            for trk_idx, det_idx in matches:
                trk = tracker_list[trk_idx]
                det = valid_dets[det_idx]
                trk.update(det['boxs'], raw_det=det, image=img)
                
            # --- Create new ---
            for det_idx in unmatched_dets:
                det = valid_dets[det_idx]
                
                # SỬA LỖI 2: Phải gọi track_id_counts[class_name] thay vì track_id_counts
                new_trk = Tracker(track_id_counts[class_name], det['boxs'], det['label'], initial_image=img)
                tracker_list.append(new_trk)
                track_id_counts[class_name] += 1
                
            # --- Remove dead ---
            trackers[class_name] = [t for t in tracker_list if t.time_since_update <= 30]
            tracker_list = trackers[class_name]
            
            # CHỐT THỜI GIAN
            tracking_time += time.time() - start_track
            tracking_calls += 1
            
            # =========================
            # COLLECT HYPOTHESIS
            # =========================
            frame_hypos = []
            frame_hypo_ids = []
            
            for trk in tracker_list:
                if trk.time_since_update == 0:
                    box = get_box_coords(trk.kf.x.flatten()[:4])
                    x1, y1, x2, y2 = box
                    w = x2 - x1
                    h = y2 - y1
                    frame_hypos.append([x1, y1, w, h])
                    frame_hypo_ids.append(trk.id)

            # =========================
            # COLLECT GT
            # =========================
            frame_gt = []
            frame_gt_ids = []
            if not gt_df.empty:
                gt_now = gt_df[(gt_df['frame'] == k) & (gt_df['type'] == class_name)]
                for _, row in gt_now.iterrows():
                    if int(row['track_id']) == -1:
                        continue
                    w = row['bbox_r'] - row['bbox_l']
                    h = row['bbox_b'] - row['bbox_t']
                    frame_gt.append([row['bbox_l'], row['bbox_t'], w, h])
                    frame_gt_ids.append(int(row['track_id']))

            # =========================
            # UPDATE METRICS
            # =========================
            if not hasattr(np, 'asfarray'):
                np.asfarray = lambda x: np.asarray(x, dtype=float)

            dist_matrix = mm.distances.iou_matrix(frame_gt, frame_hypos, max_iou=0.5)
            accs[class_name].update(frame_gt_ids, frame_hypo_ids, dist_matrix)

    # Lưu lại bộ thu thập metrics của sequence này
    for c in TARGET_CLASSES:
        all_accs[c].append(accs[c])
        all_names[c].append(SEQ_ID)

print("\nTracking Completed.")
print("\n========== TRACKING SPEED ==========")
print(f"Total tracker calls: {tracking_calls}")
print(f"Total tracking time: {tracking_time:.3f} seconds")
print(f"Average time per call: {(tracking_time/tracking_calls)*1000:.3f} ms")


Sequences: ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020']

========== Processing Sequence 0000 ==========
Frames: 154


  0%|          | 0/154 [00:00<?, ?it/s]


========== Processing Sequence 0001 ==========
Frames: 447


  0%|          | 0/447 [00:00<?, ?it/s]


========== Processing Sequence 0002 ==========
Frames: 233


...


========== Processing Sequence 0019 ==========
Frames: 1059


  0%|          | 0/1059 [00:00<?, ?it/s]


========== Processing Sequence 0020 ==========
Frames: 837


  0%|          | 0/837 [00:00<?, ?it/s]


Tracking Completed.

========== TRACKING SPEED ==========
Total tracker calls: 24024
Total tracking time: 182.202 seconds
Average time per call: 7.584 ms


In [23]:
mh = mm.metrics.create()
all_class_summaries = []

for class_name in TARGET_CLASSES:
    summary_class = mh.compute_many(
        all_accs[class_name], 
        metrics=['num_frames', 'mota', 'motp', 'idf1', 'idp', 'idr', 'num_switches'], 
        names=all_names[class_name],
        generate_overall=True
    )

    # Quan trọng: bỏ các sequence có MOTP NaN trước khi aggregate
    if 'OVERALL' in summary_class.index:
        motp_vals = summary_class.loc[summary_class.index != 'OVERALL', 'motp']
        summary_class.loc['OVERALL', 'motp'] = motp_vals.mean(skipna=True)

    summary_class.insert(0, 'Class', class_name)
    all_class_summaries.append(summary_class)


# Ghép bảng
final_df = pd.concat(all_class_summaries)
final_df.reset_index(inplace=True)
final_df.rename(columns={'index': 'Sequence'}, inplace=True)

final_df = final_df.sort_values(by=['Sequence', 'Class'])
final_df.set_index(['Sequence', 'Class'], inplace=True)

display(final_df)

num_frames      mota      motp      idf1       idp  \
Sequence Class                                                            
0000     Car                154  0.983539  0.078319  0.991770  0.991770   
         Cyclist            154  1.000000  0.075899  1.000000  1.000000   
         Pedestrian         154  1.000000  0.174298  1.000000  1.000000   
0001     Car                447  0.959717  0.102704  0.977993  0.977993   
         Cyclist            447       NaN       NaN       NaN       NaN   
...                         ...       ...       ...       ...       ...   
0020     Cyclist            837       NaN       NaN       NaN       NaN   
         Pedestrian         837       NaN       NaN       NaN       NaN   
OVERALL  Car               8008  0.981099  0.078026  0.966227  0.966227   
         Cyclist           8008  0.985036  0.078953  0.986068  0.986068   
         Pedestrian        8008  0.985004  0.119195  0.931735  0.931735   

                          idr  num_switches  
Sequence Class                               
0000     Car         0.991770             0  
         Cyclist     1.000000             0  
         Pedestrian  1.000000             0  
0001     Car         0.977993            10  
         Cyclist          NaN             0  
...                       ...           ...  
0020     Cyclist          NaN             0  
         Pedestrian       NaN             0  
OVERALL  Car         0.966227            92  
         Cyclist     0.986068             7  
         Pedestrian  0.931735            94  

[66 rows x 7 columns]

In [24]:
save_path = "/kaggle/working/kalman(ours)_kitti_results_15_03.csv"
final_df.to_csv(save_path, float_format='%.3f')
print("Saved to:", save_path)

Saved to: /kaggle/working/kalman(ours)_kitti_results_15_03.csv
